[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/63_multi_token_prediction_loss_solution.ipynb)

# 🟡 Solution: Multi-Token Prediction Loss

Reference solution for `multi_token_prediction_loss`.

In [ ]:
# Install the latest torch-judge from this repo in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/CharlesShang/TorchCode.git@master')
except ImportError:
    pass


In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def multi_token_prediction_loss(logits: torch.Tensor, targets: torch.Tensor,
                                ignore_index: int = -100) -> torch.Tensor:
    B, S, K, V = logits.shape
    losses = []
    for k in range(K):
        tgt = targets[:, k + 1:k + 1 + S]
        logit = logits[:, :, k, :]
        mask = tgt != ignore_index
        if mask.any():
            selected_logits = logit[mask]
            selected_targets = tgt[mask]
            log_probs = selected_logits - torch.logsumexp(selected_logits, dim=-1, keepdim=True)
            losses.append(-log_probs[torch.arange(selected_targets.numel(), device=logits.device), selected_targets])
    if not losses:
        return logits.sum() * 0.0
    return torch.cat(losses).mean()


In [ ]:
# Verify
logits = torch.randn(2, 4, 3, 10)
targets = torch.randint(0, 10, (2, 7))
print(multi_token_prediction_loss(logits, targets))


In [ ]:
# Run judge
from torch_judge import check
check('multi_token_prediction_loss')
